# FAISS HNSW Index Pipeline
### Approximate Nearest Neighbour Search for Dual-Encoder Proteomics Retrieval

This notebook plugs directly into the `training_workflow_dual_encoders_instanovo` notebook.  
It expects `model_spec` and `model_pep` to already be trained and in scope (or loaded from a checkpoint below).

**Pipeline stages:**
1. Configuration — all tunable knobs in one place  
2. Embedding extraction — encode the full peptide database with `model_pep`  
3. Index construction — build an HNSW index with FAISS  
4. Query — encode query spectra with `model_spec` and retrieve top-k peptides  
5. Evaluation — Recall@k against brute-force ground truth  
6. Index persistence — save and reload the index  
7. Parameter sweep — find the best M / ef_construction / ef_search tradeoff  

---
## Cell 1 — Imports

In [ ]:
import math
import time
import os
import json
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import List, Tuple, Dict, Optional

import numpy as np
import torch
import faiss
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
from tqdm.auto import tqdm

print(f"PyTorch : {torch.__version__}")
print(f"FAISS   : {faiss.__version__}")
print(f"GPU     : {torch.cuda.is_available()}")
print(f"FAISS GPU resources available: {faiss.get_num_gpus()}")

---
## Cell 2 — Configuration

All tunable parameters live here as a single dataclass.  
Nothing else in the notebook should contain magic numbers.

In [ ]:
@dataclass
class HNSWConfig:
    # ── Embedding ─────────────────────────────────────────────────────────────
    embed_dim: int = 128          # must match model_pep output dimension
    batch_size: int = 512         # batch size for embedding extraction
    normalize: bool = True        # L2-normalize embeddings before indexing
                                  # Required for cosine similarity via inner product

    # ── HNSW graph structure ──────────────────────────────────────────────────
    M: int = 32                   # Max edges per node per layer (layer > 0)
                                  # Layer 0 uses 2*M automatically by FAISS
                                  # Range: 4–64. Higher → better recall, more RAM, slower build
                                  # Typical sweet spots: 16 (fast), 32 (balanced), 64 (high recall)

    ef_construction: int = 200    # Candidate list size during index build
                                  # Higher → better graph quality, slower build
                                  # Must be >= M. Typical: 100–400
                                  # Does NOT affect query speed once built

    ef_search: int = 128          # Candidate list size during querying (runtime tunable!)
                                  # Higher → better recall, slower queries
                                  # Must be >= k. Can be changed after index is built
                                  # Typical: 64–512 depending on recall target

    # ── Retrieval ─────────────────────────────────────────────────────────────
    k_retrieve: int = 100         # How many peptide candidates to return per spectrum
    k_eval: List[int] = None      # k values to report Recall@k for

    # ── Metric space ──────────────────────────────────────────────────────────
    metric: str = "cosine"        # "cosine" (requires normalize=True) or "l2"
                                  # HNSW in FAISS only supports L2 natively;
                                  # we achieve cosine by normalizing + using
                                  # inner product (IndexHNSWFlat with METRIC_INNER_PRODUCT)

    # ── Persistence ───────────────────────────────────────────────────────────
    index_dir: str = "./hnsw_index"   # directory for saving index + metadata
    index_name: str = "peptide_hnsw"  # filename prefix

    # ── Device ────────────────────────────────────────────────────────────────
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    def __post_init__(self):
        if self.k_eval is None:
            self.k_eval = [1, 5, 10, 50, 100]
        assert self.ef_construction >= self.M, \
            f"ef_construction ({self.ef_construction}) must be >= M ({self.M})"
        assert self.ef_search >= max(self.k_eval), \
            f"ef_search ({self.ef_search}) must be >= max(k_eval) ({max(self.k_eval)})"

    def summary(self):
        print("\n── HNSW Configuration ───────────────────────────────")
        for k, v in asdict(self).items():
            print(f"  {k:<22}: {v}")
        print("─────────────────────────────────────────────────────\n")


# Instantiate — edit any field here to tune behaviour
cfg = HNSWConfig(
    embed_dim        = 128,
    batch_size       = 512,
    normalize        = True,
    M                = 32,
    ef_construction  = 200,
    ef_search        = 128,
    k_retrieve       = 100,
    k_eval           = [1, 5, 10, 50, 100],
    metric           = "cosine",
    index_dir        = "./hnsw_index",
    index_name       = "peptide_hnsw",
)
cfg.summary()

---
## Cell 3 — Model loading (skip if models are already in scope)

If you are running this notebook standalone (not inline after training),  
load your saved checkpoints here. Otherwise skip this cell.

In [ ]:
# ── Option A: models already in scope from the training notebook ──────────────
# Nothing to do. model_spec and model_pep are already defined.

# ── Option B: load from saved checkpoints ─────────────────────────────────────
# Uncomment and edit paths as needed.

# SPEC_CKPT = "./checkpoints/model_spec.pt"
# PEP_CKPT  = "./checkpoints/model_pep.pt"

# model_spec.load_state_dict(torch.load(SPEC_CKPT, map_location=cfg.device))
# model_pep.load_state_dict(torch.load(PEP_CKPT,  map_location=cfg.device))

DEVICE = torch.device(cfg.device)
model_spec = model_spec.to(DEVICE).eval()
model_pep  = model_pep.to(DEVICE).eval()

print(f"Models on device: {DEVICE}")
print(f"model_spec params: {sum(p.numel() for p in model_spec.parameters()):,}")
print(f"model_pep  params: {sum(p.numel() for p in model_pep.parameters()):,}")

---
## Cell 4 — Embedding extraction utilities

In [ ]:
def extract_peptide_embeddings(
    model_pep: torch.nn.Module,
    pep_tensor: torch.Tensor,           # (N, seq_len) int64 token ids
    batch_size: int = 512,
    device: torch.device = DEVICE,
    normalize: bool = True,
    desc: str = "Encoding peptides",
) -> np.ndarray:
    """
    Encode all peptides in batches and return a float32 numpy array of shape (N, D).

    Normalisation (L2) is applied here so the FAISS index can use inner-product
    distance as a proxy for cosine similarity.  If you index raw embeddings,
    set normalize=False and use cfg.metric='l2'.
    """
    model_pep.eval()
    all_embeddings = []
    N = pep_tensor.shape[0]

    with torch.no_grad():
        for start in tqdm(range(0, N, batch_size), desc=desc):
            batch = pep_tensor[start : start + batch_size].to(device)
            z = model_pep(batch)                         # (B, D)
            if normalize:
                z = torch.nn.functional.normalize(z, dim=-1)
            all_embeddings.append(z.cpu().float().numpy())

    return np.vstack(all_embeddings)   # (N, D)


def extract_spectrum_embeddings(
    model_spec: torch.nn.Module,
    spec_tensor: torch.Tensor,          # (N, max_peaks, 2)
    pre_tensor: torch.Tensor,           # (N, precursor_features)
    batch_size: int = 512,
    device: torch.device = DEVICE,
    normalize: bool = True,
    desc: str = "Encoding spectra",
) -> np.ndarray:
    """
    Encode all query spectra in batches.  Returns float32 numpy array (N, D).
    Normalisation must match whatever was used for the peptide index.
    """
    model_spec.eval()
    all_embeddings = []
    N = spec_tensor.shape[0]

    with torch.no_grad():
        for start in tqdm(range(0, N, batch_size), desc=desc):
            sp = spec_tensor[start : start + batch_size].to(device)
            pr = pre_tensor[start : start + batch_size].to(device)
            z  = model_spec(sp, pr)                      # (B, D)
            if normalize:
                z = torch.nn.functional.normalize(z, dim=-1)
            all_embeddings.append(z.cpu().float().numpy())

    return np.vstack(all_embeddings)   # (N, D)

---
## Cell 5 — Extract embeddings from the dataset

We build the **peptide** side into the HNSW index (the database).  
Spectrum embeddings are query vectors at search time.

In [ ]:
# ── These tensors come from the training notebook ─────────────────────────────
# train_dataset.peps  : (N_train, seq_len)      int64 peptide token ids
# train_dataset.specs : (N_train, max_peaks, 2) float32 spectra
# train_dataset.pres  : (N_train, precursor_d)  float32 precursor features
#
# For the retrieval database we use the TRAINING peptides.
# Queries come from the TEST spectra (true spectrum→peptide pairs).

print("Extracting peptide database embeddings (train split)...")
t0 = time.time()
pep_embeddings = extract_peptide_embeddings(
    model_pep,
    train_dataset.peps,
    batch_size = cfg.batch_size,
    device     = DEVICE,
    normalize  = cfg.normalize,
)
print(f"  → pep_embeddings shape : {pep_embeddings.shape}  [{time.time()-t0:.1f}s]")

print("\nExtracting query spectrum embeddings (test split)...")
t0 = time.time()
spec_embeddings = extract_spectrum_embeddings(
    model_spec,
    test_dataset.specs,
    test_dataset.pres,
    batch_size = cfg.batch_size,
    device     = DEVICE,
    normalize  = cfg.normalize,
)
print(f"  → spec_embeddings shape: {spec_embeddings.shape}  [{time.time()-t0:.1f}s]")

# Quick sanity check: norms should all be ≈ 1.0 when normalize=True
if cfg.normalize:
    pep_norms  = np.linalg.norm(pep_embeddings,  axis=1)
    spec_norms = np.linalg.norm(spec_embeddings, axis=1)
    print(f"\n  pep  norms: min={pep_norms.min():.4f}  max={pep_norms.max():.4f}")
    print(f"  spec norms: min={spec_norms.min():.4f}  max={spec_norms.max():.4f}")

---
## Cell 6 — HNSW Index builder

In [ ]:
class HNSWIndex:
    """
    Thin, well-documented wrapper around faiss.IndexHNSWFlat.

    FAISS HNSW notes
    ────────────────
    • IndexHNSWFlat stores the raw vectors ("Flat" = no quantisation).
      This gives exact distance computation within the approximate graph traversal.

    • METRIC_INNER_PRODUCT + L2-normalised vectors  ≡  cosine similarity.
      The returned distances are then in [-1, 1], higher = more similar.

    • METRIC_L2 uses squared L2 distance.  Lower = more similar.

    • FAISS HNSW does NOT support GPU natively for HNSW (only IVF indices do).
      We run CPU HNSW but use GPU for the embedding extraction step.

    • hnsw.efConstruction is set at build time and cannot be changed afterwards.
    • hnsw.efSearch    is runtime-tunable — call set_ef_search() any time.
    """

    def __init__(self, cfg: HNSWConfig):
        self.cfg     = cfg
        self.index   = None
        self.id_map  = None    # numpy int64 array: faiss_row_id → dataset_index
        self._built  = False

    # ─── Build ────────────────────────────────────────────────────────────────

    def build(self, embeddings: np.ndarray, ids: Optional[np.ndarray] = None) -> "HNSWIndex":
        """
        Build the HNSW index from a (N, D) float32 embedding matrix.

        Parameters
        ----------
        embeddings : np.ndarray, shape (N, D), dtype float32
            L2-normalised peptide embeddings (if cfg.metric == 'cosine').
        ids : optional np.ndarray of int64, shape (N,)
            External integer IDs to associate with each row.
            Defaults to 0..N-1.  Used to map search results back to
            peptide sequences / dataset rows.

        Returns self for chaining.
        """
        assert embeddings.dtype == np.float32, \
            "FAISS requires float32 — cast embeddings before calling build()."
        assert embeddings.ndim == 2 and embeddings.shape[1] == self.cfg.embed_dim, \
            f"Expected shape (N, {self.cfg.embed_dim}), got {embeddings.shape}"

        N, D = embeddings.shape
        self.id_map = ids if ids is not None else np.arange(N, dtype=np.int64)

        # ── Choose FAISS metric ────────────────────────────────────────────────
        if self.cfg.metric == "cosine":
            # Inner product on L2-normalised vectors == cosine similarity
            faiss_metric = faiss.METRIC_INNER_PRODUCT
        else:
            faiss_metric = faiss.METRIC_L2

        # ── Create index ───────────────────────────────────────────────────────
        #   IndexHNSWFlat(dim, M, metric)
        #   M  : number of bi-directional links per node per layer
        #        layer-0 gets M0 = 2*M links automatically
        self.index = faiss.IndexHNSWFlat(D, self.cfg.M, faiss_metric)

        # ef_construction: candidate list size during graph construction
        # Larger → better graph, slower build, more RAM during build only
        self.index.hnsw.efConstruction = self.cfg.ef_construction

        # ef_search: candidate list size at query time (can be changed later)
        self.index.hnsw.efSearch = self.cfg.ef_search

        # ── Add vectors ────────────────────────────────────────────────────────
        print(f"Building HNSW index: N={N:,}  D={D}  M={self.cfg.M}  "
              f"ef_construction={self.cfg.ef_construction}")
        t0 = time.time()
        self.index.add(embeddings)      # FAISS copies the array internally
        elapsed = time.time() - t0

        print(f"  Built in {elapsed:.1f}s  "
              f"({N/elapsed:,.0f} vectors/s)  "
              f"ntotal={self.index.ntotal:,}")

        # Approximate RAM used by the graph (edges only, not vectors)
        # Each node stores up to M0 (layer 0) + M * n_upper_layers int32 links
        # Very rough estimate: N * M * 4 bytes * 2 (both directions) * ~log(N) layers
        n_layers = max(1, int(math.log(N) / math.log(self.cfg.M)))
        ram_mb   = N * self.cfg.M * 4 * 2 * n_layers / 1e6
        print(f"  Estimated graph RAM : ~{ram_mb:.0f} MB  (vectors not counted)")

        self._built = True
        return self

    # ─── Runtime parameter tuning ─────────────────────────────────────────────

    def set_ef_search(self, ef_search: int):
        """
        Change the query-time candidate list size without rebuilding.
        Larger ef_search → better recall, slower queries.
        """
        assert ef_search >= 1
        self.cfg.ef_search = ef_search
        self.index.hnsw.efSearch = ef_search
        print(f"ef_search updated to {ef_search}")

    # ─── Search ───────────────────────────────────────────────────────────────

    def search(
        self,
        query_embeddings: np.ndarray,   # (Q, D) float32
        k: Optional[int] = None,
    ) -> Tuple[np.ndarray, np.ndarray]:
        """
        Retrieve the k nearest peptides for each query spectrum.

        Returns
        -------
        distances : (Q, k) float32  — cosine sim (higher=better) or L2 (lower=better)
        indices   : (Q, k) int64    — row indices into the original peptide database
        """
        assert self._built, "Call build() first."
        assert query_embeddings.dtype == np.float32

        k = k or self.cfg.k_retrieve
        distances, faiss_ids = self.index.search(query_embeddings, k)

        # Map FAISS internal row ids → original dataset indices
        indices = self.id_map[faiss_ids]   # handles -1 (not found) gracefully via id_map
        return distances, indices

    # ─── Persistence ──────────────────────────────────────────────────────────

    def save(self):
        """Save the FAISS index and id_map to cfg.index_dir."""
        Path(self.cfg.index_dir).mkdir(parents=True, exist_ok=True)
        index_path  = os.path.join(self.cfg.index_dir, self.cfg.index_name + ".faiss")
        id_map_path = os.path.join(self.cfg.index_dir, self.cfg.index_name + "_idmap.npy")
        cfg_path    = os.path.join(self.cfg.index_dir, self.cfg.index_name + "_cfg.json")

        faiss.write_index(self.index, index_path)
        np.save(id_map_path, self.id_map)
        with open(cfg_path, "w") as f:
            json.dump(asdict(self.cfg), f, indent=2)

        print(f"Index saved to {self.cfg.index_dir}/")
        print(f"  {self.cfg.index_name}.faiss  "
              f"({os.path.getsize(index_path)/1e6:.1f} MB)")

    @classmethod
    def load(cls, index_dir: str, index_name: str) -> "HNSWIndex":
        """Reload a previously saved index."""
        cfg_path    = os.path.join(index_dir, index_name + "_cfg.json")
        index_path  = os.path.join(index_dir, index_name + ".faiss")
        id_map_path = os.path.join(index_dir, index_name + "_idmap.npy")

        with open(cfg_path) as f:
            cfg_dict = json.load(f)
        cfg = HNSWConfig(**cfg_dict)

        wrapper = cls(cfg)
        wrapper.index  = faiss.read_index(index_path)
        wrapper.id_map = np.load(id_map_path)
        wrapper._built = True

        print(f"Loaded index: ntotal={wrapper.index.ntotal:,}  "
              f"efSearch={wrapper.index.hnsw.efSearch}")
        return wrapper

    # ─── Info ─────────────────────────────────────────────────────────────────

    def info(self):
        if not self._built:
            print("Index not built yet.")
            return
        print(f"\n── HNSW Index info ──────────────────────────────────")
        print(f"  Vectors indexed   : {self.index.ntotal:,}")
        print(f"  Dimension         : {self.cfg.embed_dim}")
        print(f"  M (max edges)     : {self.cfg.M}")
        print(f"  ef_construction   : {self.cfg.ef_construction}")
        print(f"  ef_search (now)   : {self.index.hnsw.efSearch}")
        print(f"  Metric            : {self.cfg.metric}")
        print(f"  Max layer         : {self.index.hnsw.max_level}")
        print("─────────────────────────────────────────────────────\n")

---
## Cell 7 — Build the index

In [ ]:
# Ensure float32 — FAISS requirement
pep_embeddings_f32  = pep_embeddings.astype(np.float32)
spec_embeddings_f32 = spec_embeddings.astype(np.float32)

hnsw_index = HNSWIndex(cfg)
hnsw_index.build(pep_embeddings_f32)
hnsw_index.info()

---
## Cell 8 — Recall@k evaluation

For each test spectrum we know the ground-truth peptide index (diagonal pairing).  
We check whether the HNSW search returns it within the top-k results.

In [ ]:
def evaluate_recall(
    hnsw_index: HNSWIndex,
    query_embeddings: np.ndarray,          # (Q, D) float32  — spectrum embeddings
    ground_truth_ids: np.ndarray,          # (Q,)   int64    — correct peptide row idx
    k_vals: List[int],
    batch_size: int = 1024,
) -> Dict[int, float]:
    """
    Compute Recall@k for each k in k_vals.

    Recall@k = fraction of queries where the correct peptide appears
               in the top-k returned by the index.

    Parameters
    ----------
    ground_truth_ids : The row index into the peptide database that
                       corresponds to each query spectrum.  For the
                       paired train/test dataset this is simply
                       np.arange(N_test) if peptides and spectra are
                       aligned, or the explicit pairing index array.
    """
    max_k = max(k_vals)
    hits  = {k: 0 for k in k_vals}
    Q     = query_embeddings.shape[0]

    for start in tqdm(range(0, Q, batch_size), desc="Evaluating Recall@k"):
        q_batch  = query_embeddings[start : start + batch_size]
        gt_batch = ground_truth_ids[start : start + batch_size]

        _, idx_batch = hnsw_index.search(q_batch, k=max_k)   # (B, max_k)

        for k in k_vals:
            # Check if gt appears in the first k columns
            in_top_k = (idx_batch[:, :k] == gt_batch[:, None]).any(axis=1)
            hits[k] += in_top_k.sum()

    recall = {k: hits[k] / Q for k in k_vals}
    return recall


# ── Ground truth: test spectrum i pairs with peptide i in the training database ──
# Adjust this if your pairing is not an identity mapping.
# For the ms_ninespecies benchmark, spectra and peptides are aligned row-by-row.
ground_truth_ids = np.arange(len(spec_embeddings_f32), dtype=np.int64)

print(f"Evaluating with ef_search={cfg.ef_search} ...\n")
recall_results = evaluate_recall(
    hnsw_index,
    spec_embeddings_f32,
    ground_truth_ids,
    k_vals     = cfg.k_eval,
    batch_size = 1024,
)

print("\n── Recall@k results ─────────────────────────────────")
for k, r in recall_results.items():
    bar = "█" * int(r * 40)
    print(f"  Recall@{k:<4d}: {r:.4f}  {bar}")
print("─────────────────────────────────────────────────────")

---
## Cell 9 — Compare HNSW vs Brute-Force (exact) baseline

In [ ]:
def build_exact_index(
    embeddings: np.ndarray,
    metric: str = "cosine",
) -> faiss.Index:
    """
    Build a brute-force IndexFlatIP (cosine) or IndexFlatL2 baseline.
    O(N*D) per query — exact but slow.  Used to verify HNSW recall.
    """
    D = embeddings.shape[1]
    if metric == "cosine":
        idx = faiss.IndexFlatIP(D)
    else:
        idx = faiss.IndexFlatL2(D)
    idx.add(embeddings)
    return idx


# Build exact index
print("Building exact (brute-force) index for comparison...")
exact_index = build_exact_index(pep_embeddings_f32, metric=cfg.metric)
print(f"  Exact index ntotal: {exact_index.ntotal:,}")

# Time a batch of 200 queries on exact vs HNSW
N_BENCH = min(200, spec_embeddings_f32.shape[0])
q_bench = spec_embeddings_f32[:N_BENCH]

t0 = time.time()
exact_index.search(q_bench, cfg.k_retrieve)
t_exact = (time.time() - t0) / N_BENCH * 1000

t0 = time.time()
hnsw_index.search(q_bench, cfg.k_retrieve)
t_hnsw = (time.time() - t0) / N_BENCH * 1000

print(f"\n── Latency per query (k={cfg.k_retrieve}) ─────────────")
print(f"  Exact (brute force) : {t_exact:.3f} ms")
print(f"  HNSW (ef={cfg.ef_search})    : {t_hnsw:.3f} ms")
print(f"  Speedup             : {t_exact/t_hnsw:.1f}×")

---
## Cell 10 — ef_search sweep: recall vs latency tradeoff

In [ ]:
# ── Sweep ef_search without rebuilding the index ──────────────────────────────
# This is the key runtime knob: change it to trade recall for speed.

EF_VALUES    = [16, 32, 64, 128, 256, 512]
K_FOR_RECALL = 10      # Recall@10 for the sweep
N_SWEEP      = min(500, spec_embeddings_f32.shape[0])
q_sweep      = spec_embeddings_f32[:N_SWEEP]
gt_sweep     = ground_truth_ids[:N_SWEEP]

sweep_results = []

for ef in EF_VALUES:
    hnsw_index.set_ef_search(ef)

    # Recall@K_FOR_RECALL
    _, idx_s = hnsw_index.search(q_sweep, k=K_FOR_RECALL)
    recall = (idx_s == gt_sweep[:, None]).any(axis=1).mean()

    # Latency (average over N_SWEEP queries)
    t0 = time.time()
    hnsw_index.search(q_sweep, k=K_FOR_RECALL)
    latency_ms = (time.time() - t0) / N_SWEEP * 1000

    sweep_results.append({"ef_search": ef, "recall": recall, "latency_ms": latency_ms})
    print(f"  ef_search={ef:4d}  Recall@{K_FOR_RECALL}={recall:.4f}  {latency_ms:.3f} ms/query")

# Reset to configured value
hnsw_index.set_ef_search(cfg.ef_search)

# ── Plot ───────────────────────────────────────────────────────────────────────
df_sweep = pd.DataFrame(sweep_results)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f"ef_search sweep  (M={cfg.M}, ef_construction={cfg.ef_construction})",
             fontsize=13)

ax = axes[0]
ax.plot(df_sweep["ef_search"], df_sweep["recall"], "o-", color="steelblue", lw=2)
ax.set_xlabel("ef_search"); ax.set_ylabel(f"Recall@{K_FOR_RECALL}")
ax.set_title("Recall vs ef_search")
ax.set_ylim(0, 1.05); ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mticker.ScalarFormatter())

ax = axes[1]
ax.plot(df_sweep["latency_ms"], df_sweep["recall"], "o-", color="coral", lw=2)
for _, row in df_sweep.iterrows():
    ax.annotate(f"ef={int(row['ef_search'])}",
                (row["latency_ms"], row["recall"]),
                textcoords="offset points", xytext=(4, 4), fontsize=8)
ax.set_xlabel("Latency (ms/query)"); ax.set_ylabel(f"Recall@{K_FOR_RECALL}")
ax.set_title("Recall vs Latency  (operating curve)")
ax.set_ylim(0, 1.05); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(cfg.index_dir, "ef_search_sweep.png"), dpi=150, bbox_inches="tight")
plt.show()

---
## Cell 11 — M and ef_construction sweep (full rebuild, run once)

These control graph quality and cannot be changed after build.  
Run this cell once to find your best M / ef_construction combination,  
then fix the values in `HNSWConfig` and rebuild.

In [ ]:
# ── WARNING: rebuilds the index multiple times — can be slow ──────────────────
# Reduce N_SUBSET or M_VALS/EF_VALS if this takes too long.

N_SUBSET   = min(50_000, pep_embeddings_f32.shape[0])  # use a subset for the sweep
M_VALS     = [8, 16, 32]
EF_C_VALS  = [64, 200]
EF_S_FIXED = 64
K_SWEEP    = 10

pep_sub  = pep_embeddings_f32[:N_SUBSET]
spec_sub = spec_embeddings_f32[:N_SUBSET]
gt_sub   = ground_truth_ids[:N_SUBSET]

grid_results = []
print(f"{'M':>4}  {'ef_c':>6}  {'Recall@'+str(K_SWEEP):>12}  {'Build(s)':>10}  {'Query(ms)':>10}")
print("-" * 50)

for M_val in M_VALS:
    for ef_c in EF_C_VALS:
        sweep_cfg = HNSWConfig(
            embed_dim       = cfg.embed_dim,
            M               = M_val,
            ef_construction = ef_c,
            ef_search       = EF_S_FIXED,
            k_retrieve      = K_SWEEP,
            k_eval          = [K_SWEEP],
            metric          = cfg.metric,
            normalize       = cfg.normalize,
            index_dir       = cfg.index_dir,
            index_name      = "sweep_tmp",
        )
        t0    = time.time()
        idx   = HNSWIndex(sweep_cfg).build(pep_sub)
        t_build = time.time() - t0

        t0 = time.time()
        _, retrieved = idx.search(spec_sub, k=K_SWEEP)
        t_q = (time.time() - t0) / N_SUBSET * 1000

        recall = (retrieved == gt_sub[:, None]).any(axis=1).mean()
        print(f"{M_val:>4}  {ef_c:>6}  {recall:>12.4f}  {t_build:>10.1f}  {t_q:>10.3f}")
        grid_results.append(dict(M=M_val, ef_c=ef_c, recall=recall,
                                 build_s=t_build, query_ms=t_q))

best = max(grid_results, key=lambda x: x["recall"])
print(f"\nBest config: M={best['M']}  ef_construction={best['ef_c']}  "
      f"→ Recall@{K_SWEEP}={best['recall']:.4f}")

---
## Cell 12 — Save the index

In [ ]:
hnsw_index.save()

---
## Cell 13 — Reload and verify

In [ ]:
# Load index from disk (standalone usage, e.g. inference notebook)
loaded_index = HNSWIndex.load(cfg.index_dir, cfg.index_name)
loaded_index.info()

# Quick sanity check
dists, idxs = loaded_index.search(spec_embeddings_f32[:5], k=3)
print("Top-3 retrieved peptide indices for first 5 queries:")
print(idxs)
print("Cosine similarities:")
print(np.round(dists, 4))

---
## Cell 14 — Visualise retrieval quality per query

Histogram of the rank at which the correct peptide was found.

In [ ]:
N_VIZ   = min(5000, spec_embeddings_f32.shape[0])
q_viz   = spec_embeddings_f32[:N_VIZ]
gt_viz  = ground_truth_ids[:N_VIZ]

_, idx_viz = hnsw_index.search(q_viz, k=cfg.k_retrieve)

# For each query, find the rank of the correct peptide (-1 = not found)
ranks = []
for i in range(N_VIZ):
    hits = np.where(idx_viz[i] == gt_viz[i])[0]
    ranks.append(hits[0] + 1 if len(hits) > 0 else cfg.k_retrieve + 1)
ranks = np.array(ranks)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle(f"Retrieval rank distribution  (N={N_VIZ}, k={cfg.k_retrieve})", fontsize=13)

# Histogram of ranks (found)
found_ranks = ranks[ranks <= cfg.k_retrieve]
not_found   = (ranks > cfg.k_retrieve).sum()

ax = axes[0]
ax.hist(found_ranks, bins=min(50, cfg.k_retrieve), color="steelblue", edgecolor="white", lw=0.3)
ax.set_xlabel("Rank of correct peptide")
ax.set_ylabel("Number of queries")
ax.set_title(f"Rank histogram  ({not_found} not found in top-{cfg.k_retrieve})")
ax.grid(True, alpha=0.3)

# Cumulative recall curve
ax = axes[1]
k_axis  = np.arange(1, cfg.k_retrieve + 1)
cum_rec = [(ranks <= k).mean() for k in k_axis]
ax.plot(k_axis, cum_rec, lw=2, color="coral")
for k_mark in cfg.k_eval:
    r = (ranks <= k_mark).mean()
    ax.axvline(k_mark, ls="--", lw=0.8, color="gray")
    ax.text(k_mark + 0.5, r - 0.03, f"R@{k_mark}\n{r:.3f}", fontsize=8)
ax.set_xlabel("k"); ax.set_ylabel("Recall@k")
ax.set_title("Cumulative Recall curve")
ax.set_ylim(0, 1.05); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(cfg.index_dir, "retrieval_rank_dist.png"), dpi=150, bbox_inches="tight")
plt.show()

---
## Cell 15 — Incremental updates

FAISS HNSW supports `add()` after build — new vectors are inserted into the graph  
exactly as during the original construction phase.  
Removal is **not** supported natively; rebuild if you need deletes.

In [ ]:
def add_to_index(
    hnsw_index: HNSWIndex,
    model_pep: torch.nn.Module,
    new_pep_tensor: torch.Tensor,      # (N_new, seq_len) int64
    new_ids: Optional[np.ndarray] = None,
    batch_size: int = 512,
    device: torch.device = DEVICE,
    normalize: bool = True,
) -> HNSWIndex:
    """
    Encode new peptides and add them to an existing HNSW index.

    new_ids : if provided, must be globally unique and not overlap with
              existing id_map values. Defaults to continuing from the
              current max id.
    """
    new_emb = extract_peptide_embeddings(
        model_pep, new_pep_tensor,
        batch_size=batch_size, device=device, normalize=normalize,
        desc="Encoding new peptides",
    ).astype(np.float32)

    N_old = hnsw_index.index.ntotal
    N_new = new_emb.shape[0]

    if new_ids is None:
        new_ids = np.arange(N_old, N_old + N_new, dtype=np.int64)

    hnsw_index.index.add(new_emb)
    hnsw_index.id_map = np.concatenate([hnsw_index.id_map, new_ids])

    print(f"Added {N_new:,} vectors.  New ntotal: {hnsw_index.index.ntotal:,}")
    return hnsw_index


# Example usage (commented out — only run when you have new peptides):
# new_pep_tokens = ...   # (N_new, seq_len) tensor
# hnsw_index = add_to_index(hnsw_index, model_pep, new_pep_tokens)

---
## Cell 16 — End-to-end inference helper

Single function you call at inference time: raw spectrum → ranked peptide list.

In [ ]:
@torch.no_grad()
def retrieve_peptides(
    spectrum: np.ndarray,               # (max_peaks, 2) float32 — single spectrum
    precursor: np.ndarray,              # (precursor_dim,) float32 — precursor features
    model_spec: torch.nn.Module,
    hnsw_index: HNSWIndex,
    peptide_sequences: List[str],       # full peptide string list aligned to the index
    k: int = 10,
    device: torch.device = DEVICE,
) -> List[Tuple[str, float]]:
    """
    Given a single spectrum, return the top-k peptide candidates
    with their cosine similarity scores.

    Returns: [(peptide_sequence, cosine_score), ...] sorted best-first.
    """
    model_spec.eval()

    # Add batch dimension
    spec_t = torch.tensor(spectrum, dtype=torch.float32).unsqueeze(0).to(device)   # (1, P, 2)
    pre_t  = torch.tensor(precursor, dtype=torch.float32).unsqueeze(0).to(device)  # (1, D)

    z = model_spec(spec_t, pre_t)                              # (1, embed_dim)
    z = torch.nn.functional.normalize(z, dim=-1)
    z_np = z.cpu().float().numpy()                             # (1, embed_dim)

    distances, indices = hnsw_index.search(z_np, k=k)         # (1, k)

    results = [
        (peptide_sequences[int(idx)], float(dist))
        for idx, dist in zip(indices[0], distances[0])
        if idx >= 0
    ]
    return results


# ── Quick smoke test ───────────────────────────────────────────────────────────
# Use train_dataset.sequences if available (list of peptide strings)
if hasattr(train_dataset, 'sequences'):
    sample_spec = test_dataset.specs[0].numpy()   # (max_peaks, 2)
    sample_pre  = test_dataset.pres[0].numpy()    # (precursor_dim,)

    candidates = retrieve_peptides(
        sample_spec, sample_pre,
        model_spec, hnsw_index,
        train_dataset.sequences,
        k=10,
    )

    print("Top-10 candidates for query spectrum 0:")
    for rank, (seq, score) in enumerate(candidates, 1):
        print(f"  {rank:>2}. {seq:<42}  cosine={score:.4f}")